# NB_CAT_01 · Armar el linaje

Une lo que ya guardan los otros casos en un solo grafo **origen → bronze → silver → gold → modelo semántico → reporte**:

| De dónde | Qué aporta |
|---|---|
| `meta.ENTITY`, `meta.ENTITY_COLUMN` (pipeline medallion) | Origen, bronze y silver de cada entidad, y sus columnas PII |
| `meta.GOLD_OBJECT` | De qué tablas lee cada objeto gold (el `SQL_TEXT` se analiza con el parser de Spark, sin ejecutarlo) |
| `Files/doc_modelos/*/modelo.json` (Documentar_Modelo_Semantico) | Tablas de cada modelo semántico, de qué tabla de gold lee cada una y qué columnas parecen PII |
| Reportes del workspace (`sempy`) | Qué reporte usa qué modelo |
| `config.DOMAIN_OBJECTS` (inventario), `perfil.tabla`, `doc.diccionario`, `cat.ACTIVO` | Items de Fabric, filas y descripciones, dueños y criticidad |

Recibe la metadata por parámetro (`fuentes_json`, salida de `cat.usp_GET_FUENTES`): **no se conecta a la base de control**. Escribe `catalogo.nodo`, `catalogo.arista`, `catalogo.impacto` y `catalogo.hallazgo` en el lakehouse, y `Files/catalogo/linaje.md` con el diagrama. **No lee filas de datos**: solo nombres, columnas y metadata.

## Parámetros

In [ ]:
# Parámetros (celda marcada como "parameters": se pueden pasar desde un pipeline)
run_id = "manual"
fuentes_json = "{}"                               # salida de cat.usp_GET_FUENTES (FUENTES_JSON)
reportes_json = ""                                # vacío = los lee con sempy; o una lista [{"reporte": ..., "modelo": ...}]
workspace = ""                                    # workspace de los reportes; vacío = el del notebook
carpeta_modelos = "/lakehouse/default/Files/doc_modelos"
carpeta_salida = "/lakehouse/default/Files/catalogo"
esquema = "catalogo"                              # esquema de LH_MEDALLION donde quedan las tablas del linaje
max_nodos_diagrama = 120                          # con más nodos, linaje.md trae un diagrama por objeto gold

## Funciones

In [ ]:
# --- funciones (sin Fabric: las usa pruebas/prueba_local.py)
# Arman el linaje origen -> bronze -> silver -> gold -> modelo -> reporte con lo que ya
# guardan los otros casos. Solo leen metadata: nunca filas de datos.
import json
import re
from collections import defaultdict, deque

SEVERIDADES = ("ALTA", "MEDIA", "INFO")


def _sin_comentarios(sql):
    sql = re.sub(r"/\*.*?\*/", " ", sql or "", flags=re.S)
    sql = re.sub(r"--[^\n]*", " ", sql)
    return re.sub(r"'(?:[^']|'')*'", "''", sql)          # los textos entre comillas no cuentan


def cuerpo_select(sql):
    """De un CREATE MATERIALIZED LAKE VIEW ... AS SELECT deja solo el SELECT."""
    s = _sin_comentarios(sql).strip()
    if re.match(r"(?is)^CREATE\b", s):
        m = re.search(r"(?is)\bAS\s+((?:SELECT|WITH)\b.*)$", s)
        return m.group(1) if m else ""
    return s


def nombres_cte(sql):
    return {n.lower() for n in re.findall(r"(?is)(?:\bWITH|,)\s*`?(\w+)`?\s+AS\s*\(", _sin_comentarios(sql))}


def normalizar(nombre):
    """`LH_GOLD`.gold.Fact -> gold.fact. El nodo es esquema.tabla: el lakehouse se descarta."""
    partes = [p.strip("` ").lower() for p in str(nombre).split(".") if p.strip("` ")]
    return ".".join(partes[-2:])


def tablas_sql(sql, parse_tree=None):
    """Tablas que lee un SELECT de gold. Con parse_tree (el parser de Spark) es exacto;
    sin él, una expresión regular sobre FROM y JOIN. Devuelve (tablas, método)."""
    cuerpo = cuerpo_select(sql)
    ctes = nombres_cte(cuerpo)
    if parse_tree is not None:
        try:
            arbol = parse_tree(cuerpo)
            refs = [".".join(p.strip(" `") for p in grupo.split(","))
                    for grupo in re.findall(r"UnresolvedRelation \[([^\]]*)\]", arbol)]
            return sorted({normalizar(r) for r in refs if r.lower() not in ctes}), "parser"
        except Exception:
            pass
    refs = re.findall(r"(?is)\b(?:FROM|JOIN)\s+(`?\w+`?(?:\s*\.\s*`?\w+`?){0,2})", cuerpo)
    return sorted({normalizar(re.sub(r"\s", "", r)) for r in refs if r.strip("`").lower() not in ctes}), "regex"


def tablas_desde_m(expresion):
    """Tablas de un lakehouse que nombra una expresión M de importación. [verificar] el formato
    que genera Power BI Desktop: navegación con Schema="gold", Item="fact_ventas"."""
    pares = re.findall(r'Schema\s*=\s*"([^"]+)"\s*,\s*Item\s*=\s*"([^"]+)"', expresion or "", flags=re.I)
    return [normalizar(f"{e}.{i}") for e, i in pares]


def lh(nombre):
    return "lh:" + normalizar(nombre)


def capa_de(nombre):
    esquema = normalizar(nombre).split(".")[0]
    return esquema if esquema in ("bronze", "silver", "gold") else "otro"


class Linaje:
    def __init__(self):
        self.nodos, self.aristas, self.hallazgos = {}, {}, []

    def nodo(self, id_, tipo, nombre, capa=None, **attrs):
        n = self.nodos.setdefault(id_, {"id": id_, "tipo": tipo, "nombre": nombre, "capa": capa or tipo,
                                        "pii": [], "dueno": None, "area": None, "criticidad": None,
                                        "descripcion": None, "atributos": {}})
        if tipo != "tabla_lh" or n["tipo"] == "tabla_lh":
            n["tipo"] = tipo
        n["atributos"].update({k: v for k, v in attrs.items() if v is not None})
        return n

    def arista(self, desde, hacia, tipo, fuente, confianza="alta"):
        clave = (desde, hacia)
        previa = self.aristas.get(clave)
        orden = {"alta": 3, "media": 2, "baja": 1}
        if previa is None or orden[confianza] > orden[previa["confianza"]]:
            self.aristas[clave] = {"desde": desde, "hacia": hacia, "tipo": tipo, "fuente": fuente, "confianza": confianza}

    def hallazgo(self, severidad, codigo, objeto, detalle):
        self.hallazgos.append({"severidad": severidad, "codigo": codigo, "objeto": objeto, "detalle": detalle})

    def entrantes(self, id_):
        return [a["desde"] for a in self.aristas.values() if a["hacia"] == id_]

    def salientes(self, id_):
        return [a["hacia"] for a in self.aristas.values() if a["desde"] == id_]


def armar_linaje(fuentes, modelos=(), reportes=(), parse_tree=None, columnas=None, perfil=None, diccionario=None):
    """fuentes: salida de cat.usp_GET_FUENTES (entities, gold_objects, items, activos).
    modelos: los modelo.json de NB_DOC_01. reportes: [{reporte, modelo}].
    columnas(tabla) -> lista de columnas si la tabla existe en el lakehouse, o None.
    perfil: {tabla bronze: {filas, descripcion_origen}}. diccionario: {(modelo, tabla): descripción}."""
    L = Linaje()
    columnas = columnas or (lambda t: None)
    perfil, diccionario = perfil or {}, diccionario or {}

    # 1. Origen -> bronze -> silver (meta.ENTITY)
    for e in fuentes.get("entities") or []:
        activa = bool(e.get("is_active", True))
        o = f"origen:{e.get('source_system_code') or 'origen'}/{e['source_object']}"
        L.nodo(o, "origen", e["source_object"], sistema=e.get("source_system_code"), tipo_origen=e.get("source_type"),
               conexion=e.get("connection_ref"))
        b, s = lh(e["bronze_table"]), lh(e["silver_table"])
        pb = perfil.get(normalizar(e["bronze_table"])) or {}
        nb = L.nodo(b, "tabla_lh", normalizar(e["bronze_table"]), "bronze", entidad=e["entity_code"],
                    filas_perfil=pb.get("filas"), activa=activa)
        nb["descripcion"] = nb["descripcion"] or pb.get("descripcion_origen")
        ns = L.nodo(s, "tabla_lh", normalizar(e["silver_table"]), "silver", entidad=e["entity_code"],
                    silver_mode=e.get("silver_mode"), delete_mode=e.get("delete_mode"),
                    load_group=e.get("load_group"), activa=activa)
        ns["pii"] = sorted({c["target_column"] for c in (e.get("columns") or []) if c.get("is_pii")})
        L.arista(o, b, "copia", "meta.ENTITY")
        L.arista(b, s, "carga", "meta.ENTITY")

    producidas = {n["id"] for n in L.nodos.values() if n["capa"] in ("bronze", "silver")}
    golds = [g for g in (fuentes.get("gold_objects") or [])]
    por_codigo = {g["gold_code"]: lh(g["target_table"]) for g in golds}
    producidas |= set(por_codigo.values())

    # 2. Gold (meta.GOLD_OBJECT): de qué tablas lee cada SELECT
    for g in golds:
        t = por_codigo[g["gold_code"]]
        L.nodo(t, "tabla_lh", normalizar(g["target_table"]), capa_de(g["target_table"]), gold_code=g["gold_code"],
               load_mode=g.get("load_mode"), load_group=g.get("load_group"), activa=bool(g.get("is_active", True)))
        refs, metodo = tablas_sql(g.get("sql_text"), parse_tree)
        if not refs:
            L.hallazgo("MEDIA", "GOLD_SIN_FUENTES", t, f"No pude leer de qué tablas lee el SQL de {g['gold_code']}")
        deps = {d.strip() for d in (g.get("depends_on_gold") or "").split(",") if d.strip()}
        for r in refs:
            rid = "lh:" + r
            if "." not in r:
                L.hallazgo("MEDIA", "REFERENCIA_SIN_ESQUEMA", t, f"{g['gold_code']} lee '{r}' sin esquema: no se puede ubicar en el linaje")
                continue
            if rid not in producidas:
                L.nodo(rid, "tabla_lh", r, capa_de(r))
                L.hallazgo("MEDIA", "REFERENCIA_DESCONOCIDA", rid,
                           f"{g['gold_code']} lee {r}, que no carga ninguna entidad ni objeto gold de la metadata")
            L.arista(rid, t, "transforma", "SQL_TEXT", "alta" if metodo == "parser" else "media")
            # Si lee otro gold, ese gold tiene que correr antes
            codigo_ref = next((c for c, tid in por_codigo.items() if tid == rid), None)
            if codigo_ref and codigo_ref != g["gold_code"] and codigo_ref not in deps:
                L.hallazgo("ALTA", "FALTA_DEPENDS_ON_GOLD", t,
                           f"{g['gold_code']} lee {r} ({codigo_ref}) pero no lo tiene en DEPENDS_ON_GOLD: puede correr antes y leer datos viejos")
        for d in deps:
            if d in por_codigo and por_codigo[d] not in {"lh:" + r for r in refs}:
                L.hallazgo("INFO", "DEPENDENCIA_SIN_LECTURA", t, f"{g['gold_code']} depende de {d} pero su SQL no lo lee")

    # 3. Modelos semánticos (modelo.json de NB_DOC_01)
    for m in modelos or []:
        mid = f"modelo:{m['modelo']}"
        L.nodo(mid, "modelo", m["modelo"], workspace=m.get("workspace"), generado_utc=m.get("generado_utc"))
        for t in m.get("tablas") or []:
            tid = f"{mid}/{t['nombre']}"
            nt = L.nodo(tid, "tabla_modelo", t["nombre"], "modelo", modo=t.get("modo"), oculta=t.get("oculta"))
            nt["descripcion"] = diccionario.get((m["modelo"], t["nombre"])) or t.get("descripcion_actual") or None
            nt["pii"] = sorted(c["nombre"] for c in t.get("columnas") or []
                               if c.get("sensible_sugerido") and not c.get("oculta"))
            L.arista(tid, mid, "compone", "modelo.json")
            fuentes_t, confianza = [], "alta"
            for o in t.get("origen") or []:
                if o.get("entidad"):
                    fuentes_t.append(normalizar(f"{o.get('esquema') or 'dbo'}.{o['entidad']}"))
                fuentes_t += tablas_desde_m(o.get("expresion"))
            if not fuentes_t and "calculated" not in str(t.get("modo", "")).lower():
                # Sin origen en modelo.json (versión vieja de NB_DOC_01): por nombre, con confianza baja
                candidatas = [n for n in producidas if n.split(".")[-1] == t["nombre"].lower().replace(" ", "_")]
                fuentes_t, confianza = [c[3:] for c in candidatas], "baja"
            if not fuentes_t and "calculated" not in str(t.get("modo", "")).lower():
                L.hallazgo("INFO", "TABLA_MODELO_SIN_ORIGEN", tid, f"No sé de qué tabla lee {t['nombre']}: corré NB_DOC_01 de nuevo")
            for f_ in fuentes_t:
                if "lh:" + f_ not in L.nodos:
                    L.nodo("lh:" + f_, "tabla_lh", f_, capa_de(f_))
                    L.hallazgo("MEDIA", "REFERENCIA_DESCONOCIDA", "lh:" + f_,
                               f"El modelo {m['modelo']} lee {f_}, que no carga ninguna entidad ni objeto gold de la metadata")
                L.arista("lh:" + f_, tid, "lee", "modelo.json" if confianza != "baja" else "nombre", confianza)
            if nt["pii"]:
                L.hallazgo("MEDIA", "PII_EN_MODELO", tid,
                           f"Columnas visibles que parecen datos personales: {', '.join(nt['pii'])}. Ocultalas o aplicá OLS")

    # 4. Reportes
    for r in reportes or []:
        rid = f"reporte:{r['reporte']}"
        L.nodo(rid, "reporte", r["reporte"], workspace=r.get("workspace"))
        mid = f"modelo:{r['modelo']}"
        if mid not in L.nodos:
            L.nodo(mid, "modelo", r["modelo"])
            L.hallazgo("INFO", "MODELO_SIN_DOCUMENTAR", mid, f"{r['reporte']} usa {r['modelo']}, que no tiene modelo.json")
        L.arista(mid, rid, "usa", "reportes")

    # 5. Columnas reales de las tablas del lakehouse y PII que llega a gold
    for n in L.nodos.values():
        if n["tipo"] == "tabla_lh":
            cols = columnas(n["nombre"])
            if cols is not None:
                n["atributos"]["columnas"] = len(cols)
                n["atributos"]["existe"] = True
                n["_cols"] = {c.lower() for c in cols}
            elif n["capa"] in ("silver", "gold"):
                n["atributos"]["existe"] = False
    for id_ in orden_topologico(L):
        n = L.nodos[id_]
        if n["tipo"] != "tabla_lh" or n["capa"] in ("bronze", "silver"):
            continue
        heredada = set()
        for p in L.entrantes(id_):
            heredada |= {c.lower() for c in L.nodos[p]["pii"]}
        if "_cols" in n:
            expuesta = sorted(heredada & n["_cols"])
            if expuesta:
                n["pii"] = expuesta
                L.hallazgo("ALTA" if n["capa"] == "gold" else "MEDIA", "PII_EN_GOLD", id_,
                           f"{n['nombre']} tiene columnas PII que vienen de silver: {', '.join(expuesta)}")
    for n in L.nodos.values():
        n.pop("_cols", None)

    # 6. Dueños (cat.ACTIVO) e items del inventario (config.DOMAIN_OBJECTS)
    for a in fuentes.get("activos") or []:
        n = L.nodos.get(a["nodo_id"])
        if n is None:
            L.hallazgo("INFO", "ACTIVO_INEXISTENTE", a["nodo_id"], "cat.ACTIVO nombra un nodo que no está en el linaje")
            continue
        n.update(dueno=a.get("dueno"), area=a.get("area"), criticidad=a.get("criticidad"))
        n["descripcion"] = a.get("descripcion_negocio") or n["descripcion"]
    for n in L.nodos.values():
        if n["dueno"] is None and (n["capa"] == "gold" or n["tipo"] == "modelo"):
            L.hallazgo("INFO", "SIN_DUENO", n["id"], f"{n['nombre']} no tiene dueño en cat.ACTIVO")

    tipos_item = {"SemanticModel": "modelo", "Report": "reporte"}
    for it in fuentes.get("items") or []:
        tipo = tipos_item.get(it.get("artifact_type"))
        if not tipo or not it.get("active_flag", True):
            continue
        id_ = f"{tipo}:{it['object_name']}"
        if id_ in L.nodos:
            L.nodos[id_]["atributos"]["item_guid"] = it.get("fabric_item_guid")
            L.nodos[id_]["atributos"]["workspace_id"] = it.get("workspace_id")
        else:
            L.hallazgo("INFO", "ITEM_FUERA_DEL_LINAJE", id_,
                       f"{it['artifact_type']} {it['object_name']} está en el inventario pero no en el linaje")

    # 7. Lo que nadie usa
    for n in L.nodos.values():
        if n["tipo"] != "tabla_lh":
            continue
        if n["capa"] == "silver" and not L.salientes(n["id"]):
            L.hallazgo("INFO", "SILVER_SIN_USO", n["id"], f"Ningún objeto gold lee {n['nombre']}")
        if n["capa"] == "gold" and not L.salientes(n["id"]):
            L.hallazgo("INFO", "GOLD_SIN_CONSUMO", n["id"],
                       f"Ningún modelo documentado ni otro gold lee {n['nombre']} (¿lo usan las vistas del agente o un reporte sin modelo.json?)")
        if n["atributos"].get("activa") is False and L.salientes(n["id"]):
            L.hallazgo("MEDIA", "LEE_ENTIDAD_INACTIVA", n["id"],
                       f"{n['nombre']} viene de una entidad inactiva pero hay objetos que la leen: quedan con datos viejos")

    orden = {s: i for i, s in enumerate(SEVERIDADES)}
    L.hallazgos.sort(key=lambda h: (orden[h["severidad"]], h["codigo"], h["objeto"]))
    return L


def orden_topologico(L):
    grado = {i: 0 for i in L.nodos}
    for a in L.aristas.values():
        grado[a["hacia"]] += 1
    cola = deque(sorted(i for i, g in grado.items() if g == 0))
    salida = []
    while cola:
        i = cola.popleft()
        salida.append(i)
        for h in sorted(L.salientes(i)):
            grado[h] -= 1
            if grado[h] == 0:
                cola.append(h)
    if len(salida) < len(L.nodos):           # ciclo: lo informa y devuelve lo que pudo ordenar
        resto = sorted(set(L.nodos) - set(salida))
        L.hallazgo("ALTA", "CICLO", resto[0], f"Hay un ciclo en el linaje entre: {', '.join(resto[:6])}")
        salida += resto
    return salida


def impacto(L):
    """Cierre transitivo aguas abajo: (desde, hasta, distancia). Responde "¿qué toco si cambio X?"."""
    hijos = defaultdict(list)
    for a in L.aristas.values():
        hijos[a["desde"]].append(a["hacia"])
    filas = []
    for origen in L.nodos:
        vistos, cola = {origen: 0}, deque([origen])
        while cola:
            x = cola.popleft()
            for h in hijos[x]:
                if h not in vistos:
                    vistos[h] = vistos[x] + 1
                    cola.append(h)
                    filas.append({"desde": origen, "hasta": h, "distancia": vistos[h]})
    return filas


def vecindario(L, foco):
    """Ancestros y descendientes de un nodo."""
    padres, hijos = defaultdict(set), defaultdict(set)
    for a in L.aristas.values():
        padres[a["hacia"]].add(a["desde"])
        hijos[a["desde"]].add(a["hacia"])
    salida = {foco}
    for rel in (padres, hijos):
        cola = deque([foco])
        while cola:
            x = cola.popleft()
            for y in rel[x] - salida:
                salida.add(y)
                cola.append(y)
    return salida


CAPAS_MERMAID = [("origen", "Origen"), ("bronze", "Bronze"), ("silver", "Silver"), ("gold", "Gold"),
                 ("otro", "Otras tablas"), ("modelo", "Modelo semántico"), ("reporte", "Reportes")]


def mermaid(L, foco=None, max_nodos=120):
    ids = vecindario(L, foco) if foco else set(L.nodos)
    if len(ids) > max_nodos:
        return None
    alias = {i: f"n{k}" for k, i in enumerate(sorted(ids))}
    lineas = ["flowchart LR"]
    for capa, titulo in CAPAS_MERMAID:
        miembros = sorted(i for i in ids if L.nodos[i]["capa"] == capa and L.nodos[i]["tipo"] != "modelo")
        if capa == "modelo":
            miembros = sorted(i for i in ids if L.nodos[i]["tipo"] in ("modelo", "tabla_modelo"))
        if not miembros:
            continue
        lineas.append(f"    subgraph {capa}[{titulo}]")
        for i in miembros:
            n = L.nodos[i]
            texto = n["nombre"].replace('"', "'") + (" 🔒" if n["pii"] else "")
            forma = ('[("', '")]') if n["tipo"] == "tabla_lh" else (('[/"', '"/]') if n["tipo"] == "origen" else ('["', '"]'))
            lineas.append(f"        {alias[i]}{forma[0]}{texto}{forma[1]}")
        lineas.append("    end")
    for a in sorted(L.aristas.values(), key=lambda a: (a["desde"], a["hacia"])):
        if a["desde"] in ids and a["hacia"] in ids:
            flecha = "-.->" if a["confianza"] == "baja" else "-->"
            lineas.append(f"    {alias[a['desde']]} {flecha} {alias[a['hacia']]}")
    return "\n".join(lineas)


def informe_md(L, titulo="Linaje de la plataforma", max_nodos=120):
    cuenta = defaultdict(int)
    for n in L.nodos.values():
        cuenta[n["tipo"] if n["tipo"] != "tabla_lh" else n["capa"]] += 1
    md = [f"# {titulo}", "", " · ".join(f"{k}: {v}" for k, v in sorted(cuenta.items())), ""]
    graf = mermaid(L, max_nodos=max_nodos)
    md += ["## Linaje completo", ""]
    if graf:
        md += ["```mermaid", graf, "```", ""]
    else:
        md += [f"Más de {max_nodos} nodos: va un diagrama por objeto gold.", ""]
        for n in sorted(L.nodos.values(), key=lambda n: n["id"]):
            if n["capa"] == "gold" and n["tipo"] == "tabla_lh":
                g = mermaid(L, foco=n["id"], max_nodos=len(L.nodos))
                if g:
                    md += [f"### {n['nombre']}", "", "```mermaid", g, "```", ""]
    md += ["## Hallazgos", "", "| Severidad | Código | Objeto | Detalle |", "|---|---|---|---|"]
    for h in L.hallazgos:
        md.append(f"| {h['severidad']} | {h['codigo']} | `{h['objeto']}` | {h['detalle'].replace('|', '/')} |")
    if not L.hallazgos:
        md.append("| - | - | - | Sin hallazgos |")
    return "\n".join(md) + "\n"


def resumen(L):
    por_sev = defaultdict(int)
    for h in L.hallazgos:
        por_sev[h["severidad"]] += 1
    return {"nodos": len(L.nodos), "aristas": len(L.aristas), "hallazgos": dict(por_sev),
            "hay_alta": por_sev["ALTA"] > 0}

## Leer las fuentes

In [ ]:
import glob
import os

fuentes = json.loads(fuentes_json) if fuentes_json and fuentes_json.strip() else {}

modelos = []
for ruta in sorted(glob.glob(os.path.join(carpeta_modelos, "*", "modelo.json"))):
    with open(ruta, encoding="utf-8") as f:
        modelos.append(json.load(f))


def leer_reportes():
    """Reporte -> modelo semántico con sempy. [verificar] los nombres de columna de list_reports
    y list_datasets en la versión de sempy del runtime."""
    import sempy.fabric as fabric
    ws = workspace or None
    modelos_ws = fabric.list_datasets(workspace=ws)
    nombre_modelo = dict(zip(modelos_ws["Dataset ID"], modelos_ws["Dataset Name"]))
    reps = fabric.list_reports(workspace=ws)
    return [{"reporte": r["Name"], "modelo": nombre_modelo.get(r["Dataset Id"], r["Dataset Id"]), "workspace": ws}
            for _, r in reps.iterrows() if r.get("Dataset Id")]


if reportes_json and reportes_json.strip():
    reportes = json.loads(reportes_json)
else:
    try:
        reportes = leer_reportes()
    except Exception as ex:
        print("Sin reportes:", type(ex).__name__, str(ex)[:200])
        reportes = []


def columnas_de(tabla):
    try:
        return spark.table(tabla).columns if spark.catalog.tableExists(tabla) else None
    except Exception:
        return None


perfil, diccionario = {}, {}
if spark.catalog.tableExists("perfil.tabla"):
    for r in spark.sql("""SELECT tabla, filas, descripcion_origen FROM (
                              SELECT *, row_number() OVER (PARTITION BY tabla ORDER BY corrida_id DESC) AS rn
                              FROM perfil.tabla) WHERE rn = 1""").collect():
        perfil[normalizar(r["tabla"])] = {"filas": r["filas"], "descripcion_origen": r["descripcion_origen"]}
if spark.catalog.tableExists("doc.diccionario"):
    for r in spark.sql("SELECT modelo, tabla, descripcion FROM doc.diccionario WHERE tipo_objeto = 'tabla'").collect():
        if r["descripcion"] and r["descripcion"] != "[dato a confirmar]":
            diccionario[(r["modelo"], r["tabla"])] = r["descripcion"]


def arbol_spark(sql):
    """Plan sin resolver: no ejecuta ni necesita que las tablas existan."""
    return spark._jsparkSession.sessionState().sqlParser().parsePlan(sql).treeString()


print(f"{len(fuentes.get('entities') or [])} entidades, {len(fuentes.get('gold_objects') or [])} objetos gold, "
      f"{len(modelos)} modelos, {len(reportes)} reportes, {len(fuentes.get('items') or [])} items de inventario")

## Armar el linaje

In [ ]:
L = armar_linaje(fuentes, modelos, reportes, parse_tree=arbol_spark, columnas=columnas_de,
                 perfil=perfil, diccionario=diccionario)
filas_impacto = impacto(L)
res = resumen(L)
res["run_id"] = run_id
print(json.dumps(res, ensure_ascii=False))

## Guardar

In [ ]:
from datetime import datetime, timezone
from pyspark.sql import functions as F

ahora = datetime.now(timezone.utc)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {esquema}")


def guardar(filas, ddl, tabla):
    campos = [c.strip().split(" ")[0] for c in ddl.split(",")]
    df = (spark.createDataFrame([tuple(f.get(k) for k in campos) for f in filas], ddl)
          .withColumn("run_id", F.lit(run_id)).withColumn("generado_at", F.lit(ahora)))
    # Foto completa en cada corrida: el linaje de hoy reemplaza al de ayer (la historia queda en el time travel de Delta)
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{esquema}.{tabla}")
    return df.count()


nodos = [{**n, "pii": ",".join(n["pii"]) or None, "atributos": json.dumps(n["atributos"], ensure_ascii=False, default=str)}
         for n in L.nodos.values()]
n_nodos = guardar(nodos, "id string, tipo string, capa string, nombre string, dueno string, area string, "
                         "criticidad string, pii string, descripcion string, atributos string", "nodo")
guardar(list(L.aristas.values()), "desde string, hacia string, tipo string, fuente string, confianza string", "arista")
guardar(filas_impacto, "desde string, hasta string, distancia int", "impacto")
guardar(L.hallazgos, "severidad string, codigo string, objeto string, detalle string", "hallazgo")

os.makedirs(carpeta_salida, exist_ok=True)
with open(os.path.join(carpeta_salida, "linaje.md"), "w", encoding="utf-8") as f:
    f.write(informe_md(L, max_nodos=int(max_nodos_diagrama)))
with open(os.path.join(carpeta_salida, "linaje.json"), "w", encoding="utf-8") as f:
    json.dump({"generado_utc": ahora.isoformat(timespec="seconds"), "run_id": run_id,
               "nodos": nodos, "aristas": list(L.aristas.values()), "hallazgos": L.hallazgos},
              f, ensure_ascii=False, indent=1, default=str)
print(f"{n_nodos} nodos en {esquema}.nodo; diagrama en {carpeta_salida}/linaje.md")

In [ ]:
notebookutils.notebook.exit(json.dumps(res, ensure_ascii=False, default=str))